# Training

In [2]:
import pennylane as qml
import numpy as np
import torch
from qvc_model import HybridModel
from noise_models import depolarising_single_qubit, depolarising_two_qubit
import pickle
from torchvision import transforms
import torchvision

In [4]:
def job(num_qubits):
    """
    Function to run the Hybrid Quantum-Classical model. This function is called by the main script.
    """

    # Check CUDA availability
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    # Data Preprocessing

    im_size = np.sqrt(2**num_qubits).astype(int)

    #Resize images, convert to tensors
    transform = transforms.Compose([
        transforms.Resize((im_size, im_size)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.flatten()),
        transforms.Lambda(lambda x: x / torch.norm(x)),
        transforms.Lambda(lambda x: x.to(device))  # Move to device (GPU or CPU)
    ])

    # Load MNIST dataset
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Use smaller subsets for quicker training (optional)
    epoch_size = 1000 
    batch_size = 50
    test_size = 200 
    train_dataset = torch.utils.data.Subset(train_dataset, indices=range(0, len(train_dataset), 10))
    test_dataset = torch.utils.data.Subset(test_dataset, indices=range(0, len(test_dataset), 10))

    # Use DataLoader to create batches
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=test_size, shuffle=False)


    n_qubits = num_qubits
    dev=qml.device("default.mixed", wires=n_qubits)
    layers = 2
    weight_shapes = {"weights": (layers, n_qubits, 3)}  # 3 parameters per qubit for RZ, RY, RZ rotations

    # Training loop
    noise_model = depolarising_single_qubit(p_depol=0.001, p_damping=0)  # Example noise model
    #noise_model = None  # Disable noise for testing
    #phi = {f"{P}{Q}": 0.0 for P in "IXYZ" for Q in "IXYZ" if not (P == "I" and Q == "I")}
    #phi["ZZ"] = 0.00116
    model = HybridModel(dev=dev, device=device, num_qubits=n_qubits, weight_shapes=weight_shapes, num_shots=10, noise_model=noise_model).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    results = {
        'training_samples': [0],
        'accuracies': {},
        'loss_values': [],
        'gradients': [],
    }
    trained_samples = 0
    print(f"Training with {n_qubits} qubits, {layers} layers, and noise model: {noise_model}")
    n_epochs = 2  # Reduced number of epochs for quicker testing
    model.train()
    for epoch in range(n_epochs):
        
        for inputs, labels in train_loader:

            # ── Forward pass ──────────────────────────────────────────────────
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # ── NaN guard (noisy circuits occasionally produce NaN outputs) ───
            if torch.isnan(loss):
                print(f"NaN loss encountered at {trained_samples} samples, skipping batch")
                continue

            # ── Backward pass ─────────────────────────────────────────────────
            loss.backward()
            optimizer.step()

            # ── Logging ───────────────────────────────────────────────────────
            trained_samples += len(inputs)
            results["training_samples"].append(trained_samples)
            results["loss_values"].append(loss.item())

            # Gradient logging: mean squared gradient over quantum params only
            quantum_grads = optimizer.param_groups[0]["params"][0].grad
            if quantum_grads is not None:
                mean_grad_sq = quantum_grads.pow(2).mean().item()
                results["gradients"].append(mean_grad_sq)
            else:
                results["gradients"].append(float("nan"))

            #print(f"  samples {trained_samples:>6} | loss {loss.item():.4f} | "
            #      f"grad² {results['gradients'][-1]:.2e}")

        # ── Test evaluation after each epoch ──────────────────────────────────
        model.eval()
        correct = 0
        total   = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                _, predicted = torch.max(outputs, dim=1)
                correct += (predicted == labels).sum().item()
                total   += labels.size(0)

        accuracy = correct / total
        results["accuracies"][trained_samples] = accuracy
        model.train()

        print(f"Epoch {epoch+1}/{n_epochs} | "
              f"acc {accuracy:.2%} | "
              f"samples seen {trained_samples}")
    with open(f"../results/test_{num_qubits}_qubits.pkl", "wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

In [5]:
job(4)

Using device: cpu
Training with 4 qubits, 2 layers, and noise model: NoiseModel({
    OpEq(RY): DepolarizingChannel(p=0.001)
})
Epoch 1/2 | acc 19.20% | samples seen 6000
Epoch 2/2 | acc 21.50% | samples seen 12000


In [10]:
#Check version
import mpi4py

print(f"version: {torch.__version__}")

version: 2.12.1+cpu


In [ ]:
# Check that smaller MNIST dataset is still representative of the full dataset
def preprocess_MNIST_data(num_qubits: int, batch_size: int = 50, test_size: int = 200, divide_by: int = 10):
    """
    Preprocess the MNIST dataset for training and testing.
    Args:
        num_qubits: Number of qubits in the quantum circuit.
        batch_size: Batch size for training.
        test_size: Batch size for testing.
        divide_by: Factor by which to reduce the dataset size.
    Returns:
        train_loader: DataLoader for the training dataset.
        test_loader: DataLoader for the testing dataset.
    """
    im_size = int(np.sqrt(2**num_qubits))

    # Resize images, convert to tensors
    transform = transforms.Compose([
        transforms.Resize((im_size, im_size)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.flatten()),
        transforms.Lambda(lambda x: x / torch.norm(x)),
    ])

    # Load MNIST dataset
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    train_dataset = torch.utils.data.Subset(train_dataset, indices=range(0, len(train_dataset), divide_by))
    test_dataset = torch.utils.data.Subset(test_dataset, indices=range(0, len(test_dataset), divide_by))

    return train_dataset, test_dataset

In [4]:
train_dataset, test_dataset = preprocess_MNIST_data(num_qubits=4, batch_size=50, test_size=250, divide_by=4)

In [5]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 15000
Test dataset size: 2500
